**Stage 2 (thesis-era).** YOLO11n bounding-box "isolated persons" / blackout video (everything except people blacked out), as shown on the results slides. Input to the thesis-era stego notebook `notebooks/stage3_stego/Stego_and_Reconstructed_Videos.ipynb`.

In [ ]:
!pip install ultralytics opencv-python numpy

In [ ]:
# video_path = "vid4.mp4"
video_path = "vid2-2.avi"

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

def create_video_writer(cap, output_filename):
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    # fps = 30
    # fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fourcc = cv2.VideoWriter_fourcc(*'XVID') # Changed 'avi' to 'XVID'
    return cv2.VideoWriter(output_filename, fourcc, fps, (frame_width, frame_height))

def isolate_persons(frame, results):
    """Create mask showing only detected persons"""
    # Initialize black canvas
    mask = np.zeros_like(frame)

    # Copy only person regions from original frame
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
            mask[y1:y2, x1:x2] = frame[y1:y2, x1:x2]

    return mask

def blackout_person_in_frame(model, frame, classes=[0], conf=0.45):
    """Process single frame to remove detected persons using bounding boxes"""
    # Run YOLOv8 inference
    results = model.predict(frame,
                          classes=classes,
                          conf=conf,
                          verbose=False,
                          show_boxes=True)  # Ensure boxes are returned

    # Create copy of original frame
    processed_frame = frame.copy()

    # Process detections
    for result in results:
        for box in result.boxes:
            # Get bounding box coordinates
            x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())

            # Black out the detected person area
            processed_frame[y1:y2, x1:x2] = 0  # Set to black

    return processed_frame

# For isolate persons
#initialize yolo v11 model
model = YOLO("yolo11n.pt")  # Detection-only model

# Video processing setup
# video_path = "dog_walk.mp4"
# video_path = "hall-org.mp4"

cap = cv2.VideoCapture(video_path)

# Check if video opened successfully
if not cap.isOpened():
    print(f"Error: Could not open video file {video_path}")
else:
    # Correct the output filename construction
    output_video_filename = f"{video_path.split('.')[0]}-isolated_persons.avi"
    writer = create_video_writer(cap, output_video_filename)

    # Processing parameters
    PERSON_CLASS = [0]  # COCO person class ID
    CONF_THRESHOLD = 0.4  # Detection confidence

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        # Detect persons
        results = model.predict(frame,
                              classes=PERSON_CLASS,
                              conf=CONF_THRESHOLD,
                              verbose=False)

        # Create isolation mask
        processed_frame = isolate_persons(frame, results)

        # Write to output
        writer.write(processed_frame)

    # Cleanup
    cap.release()
    writer.release()
    cv2.destroyAllWindows()


# # For blackout persons

# cap = cv2.VideoCapture(video_path)
# writer = create_video_writer(cap, f"blackout_person_{video_path}")

# # Person class ID
# PERSON_CLASS = [0]

# # Processing loop
# while cap.isOpened():
#     success, frame = cap.read()
#     if not success:
#         break

#     # Process frame and remove persons
#     processed_frame = blackout_person_in_frame(model, frame,
#                                   classes=PERSON_CLASS,
#                                   conf=0.35)  # Confidence threshold

#     # Write processed frame
#     writer.write(processed_frame)

# # Cleanup
# cap.release()
# writer.release()
# cv2.destroyAllWindows()